# PyTorch — The Deep Learning Framework That Thinks Like You Do

---

## What Is PyTorch?

PyTorch is an open-source deep learning framework developed by Meta (Facebook) AI Research. Released in 2016, it has become the **dominant framework in academic research** and is rapidly gaining in production too. As of 2024, over 70% of machine learning research papers use PyTorch.

At its heart, PyTorch does two things:
1. **Tensor computation** — like NumPy, but runs on GPUs
2. **Automatic differentiation (autograd)** — automatically computes gradients so you can train neural networks without hand-deriving math

### Real-World Analogy

Think of a neural network as a very complicated recipe. NumPy is like a kitchen where every step must be done manually and you must calculate all measurements yourself. PyTorch is like a **smart kitchen** where:
- You can cook on a superfast stove (GPU)
- It automatically tracks how every ingredient affects the final dish (autograd)
- You can taste and adjust after every step (define-by-run, also called eager execution)
- You can also write the recipe down once and let the kitchen run it on autopilot (TorchScript / compile)

---

## Why Learn PyTorch?

- **Industry standard for research**: If you read a deep learning paper, the code is almost certainly in PyTorch
- **Pythonic**: Feels natural to Python programmers — debugging with `print()` just works
- **Flexible**: Build any architecture — CNNs, RNNs, Transformers, custom layers
- **Production-ready**: TorchServe, ONNX export, mobile deployment, TorchScript
- **Huge ecosystem**: HuggingFace, PyTorch Lightning, torchvision, torchaudio, torchtext

---

## Prerequisites

- Python basics (classes, functions, loops)
- NumPy fundamentals (arrays, shapes, operations)
- Basic calculus — knowing what a derivative is (not computing them manually)
- Basic ML concepts (loss function, gradient descent, training/test split)

---

## Table of Contents

1. Installation & Setup
2. Tensors — PyTorch's Core Data Structure
3. Autograd — Automatic Differentiation Explained
4. `nn.Module` — Building Neural Networks
5. The Training Loop — How PyTorch Models Learn
6. Datasets & DataLoaders — Feeding Data Efficiently
7. Saving & Loading Models
8. GPU Training
9. Convolutional Neural Networks (CNNs)
10. Recurrent Neural Networks (RNNs / LSTMs)
11. Mini Project — Handwritten Digit Classifier (MNIST)
12. Common Pitfalls
13. Interview Q&A
14. Resources
15. Summary & What's Next

---

**Official Docs:** https://pytorch.org/docs/stable/index.html  
**Tutorials:** https://pytorch.org/tutorials/  
**GitHub:** https://github.com/pytorch/pytorch  
**YouTube — PyTorch Beginner (freeCodeCamp):** https://www.youtube.com/watch?v=V_xro1bcAuA  
**YouTube — Neural Networks from Scratch (Andrej Karpathy):** https://www.youtube.com/watch?v=VMj-3S1tku0  

## 1. Installation & Setup

```bash
# CPU-only (works on any machine)
pip install torch torchvision

# With CUDA 12.1 (for NVIDIA GPU)
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
```

Visit https://pytorch.org/get-started/locally/ for the exact command for your OS and CUDA version.

In [ ]:
torchtry:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torch.optim as optim
    from torch.utils.data import Dataset, DataLoader, TensorDataset
    import numpy as np
    import matplotlib.pyplot as plt
    import warnings; warnings.filterwarnings("ignore")
    DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
    print(f"PyTorch {torch.__version__} ready | device: {DEVICE}")
except ImportError:
    raise SystemExit("Run: pip install torch torchvision")


## 2. Tensors — PyTorch's Core Data Structure

A **tensor** is an n-dimensional array — the same as a NumPy array, but:
- Can live on GPU (fast parallel computation)
- Tracks gradients for automatic differentiation

```
Scalar:  0 dimensions  →  torch.tensor(5.0)
Vector:  1 dimension   →  torch.tensor([1.0, 2.0, 3.0])
Matrix:  2 dimensions  →  torch.tensor([[1,2],[3,4]])
Tensor:  3+ dimensions →  a batch of images: shape (batch, channels, height, width)
```

In [ ]:
# ---- Creating tensors ----
t_from_list = torch.tensor([1.0, 2.0, 3.0])         # from Python list
t_zeros     = torch.zeros(3, 4)                       # 3x4 matrix of zeros
t_ones      = torch.ones(2, 3)                        # 2x3 matrix of ones
t_rand      = torch.rand(2, 3)                        # uniform [0, 1)
t_randn     = torch.randn(2, 3)                       # standard normal
t_arange    = torch.arange(0, 10, 2)                  # [0, 2, 4, 6, 8]
t_linspace  = torch.linspace(0, 1, 5)                 # [0.0, 0.25, 0.5, 0.75, 1.0]
t_from_np   = torch.from_numpy(np.array([1, 2, 3]))   # from NumPy array

print("Shape:",   t_zeros.shape)       # torch.Size([3, 4])
print("Dtype:",   t_from_list.dtype)   # torch.float32
print("Device:",  t_zeros.device)      # cpu

# ---- Tensor operations (same as NumPy) ----
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.tensor([[5.0, 6.0], [7.0, 8.0]])

print("\nElement-wise add:\n", a + b)
print("Matrix multiply:\n",   a @ b)   # same as torch.matmul(a, b)
print("Element-wise mul:\n",  a * b)
print("Sum:",                 a.sum())
print("Mean:",                a.mean())
print("Transpose:\n",         a.T)

In [ ]:
# ---- Shape manipulation — critical to understand! ----
x = torch.randn(6)
print("Original:",     x.shape)              # [6]
print("Reshaped:",     x.reshape(2, 3).shape)  # [2, 3]
print("Viewed:",       x.view(3, 2).shape)     # [3, 2]
print("Unsqueezed:",   x.unsqueeze(0).shape)   # [1, 6]  ← adds a dimension
print("Unsqueezed:",   x.unsqueeze(1).shape)   # [6, 1]  ← adds a dimension
print("Squeezed:",     x.unsqueeze(0).squeeze().shape)  # [6]  ← removes dim

# Stack and concatenate
a = torch.randn(3, 4)
b = torch.randn(3, 4)
print("\nCat dim=0:", torch.cat([a, b], dim=0).shape)   # [6, 4]
print("Cat dim=1:", torch.cat([a, b], dim=1).shape)   # [3, 8]
print("Stack:",      torch.stack([a, b], dim=0).shape)  # [2, 3, 4]  ← new dim

# Conversion
t = torch.tensor([1.0, 2.0, 3.0])
print("\nTo NumPy:", t.numpy())              # numpy array
print("Scalar:   ", t[0].item())             # Python float (use .item() for scalars!)

## 3. Autograd — Automatic Differentiation Explained

This is **the magic** of PyTorch. When you train a neural network, you need to compute gradients (derivatives) of the loss with respect to every weight. Doing this by hand for millions of parameters is impossible. PyTorch does it automatically.

### How It Works

When you do math with tensors that have `requires_grad=True`, PyTorch **records every operation** in a computation graph. When you call `.backward()`, it traverses the graph in reverse (backpropagation) and computes gradients automatically.

```
x → [operation1] → y → [operation2] → z → loss
                                           ↓ .backward()
dloss/dx ← [grad1]     dloss/dy ← [grad2] ← dloss/dz
```

In [ ]:
# ---- Simple autograd example ----
# Let's compute d/dx [ x^2 + 3x + 2 ] at x = 4
# Analytically: 2x + 3 = 2*4 + 3 = 11

x = torch.tensor(4.0, requires_grad=True)  # scalar, tracking gradients
y = x**2 + 3*x + 2                         # y = 4^2 + 3*4 + 2 = 30
print(f"y = x^2 + 3x + 2 at x=4: {y.item()}")  # 30

y.backward()   # compute dy/dx
print(f"dy/dx at x=4:    {x.grad.item()}")  # should be 11

# ---- How neural network training uses this ----
# Pretend this is a simple 1-parameter model: prediction = w * x
w = torch.tensor(2.0, requires_grad=True)   # our weight (to be learned)
x_data = torch.tensor([1.0, 2.0, 3.0, 4.0])
y_true = torch.tensor([3.0, 6.0, 9.0, 12.0])   # true: y = 3x

print("\n--- One step of gradient descent ---")
for step in range(5):
    # Forward pass: compute prediction and loss
    y_pred = w * x_data
    loss   = ((y_pred - y_true) ** 2).mean()  # MSE loss

    # Backward pass: compute gradients
    loss.backward()

    # Gradient descent update (no_grad = don't track this operation)
    with torch.no_grad():
        w -= 0.05 * w.grad   # w = w - lr * dLoss/dw

    # Zero gradients for next step (IMPORTANT! else gradients accumulate)
    w.grad.zero_()

    print(f"Step {step+1}: w={w.item():.4f}, loss={loss.item():.4f}")

print(f"\nFinal w ≈ {w.item():.4f}  (target: 3.0)")

## 4. `nn.Module` — Building Neural Networks

`torch.nn.Module` is the base class for all neural network components in PyTorch. Every layer, every model, every building block inherits from it.

**Two ways to build models:**

1. **`nn.Sequential`** — for simple, linear stacks of layers (quick and easy)
2. **Custom `nn.Module` class** — for any architecture (flexible, for complex models)

The key method to implement in a custom module: `forward(self, x)` — defines what happens during the forward pass.

In [ ]:
# ==================================================
# Method 1: nn.Sequential (simple)
# ==================================================

# A 3-layer feedforward network: 10 → 64 → 32 → 1
model_seq = nn.Sequential(
    nn.Linear(10, 64),   # fully-connected layer: 10 inputs → 64 outputs
    nn.ReLU(),           # activation: max(0, x)
    nn.Dropout(0.3),     # randomly zero 30% of neurons during training
    nn.Linear(64, 32),   # 64 → 32
    nn.ReLU(),
    nn.Linear(32, 1)     # 32 → 1 output (regression)
)

x_test = torch.randn(8, 10)  # batch of 8 samples, 10 features each
print("Sequential model output shape:", model_seq(x_test).shape)  # [8, 1]

# Count parameters
total_params = sum(p.numel() for p in model_seq.parameters())
print(f"Total parameters: {total_params:,}")

In [ ]:
# ==================================================
# Method 2: Custom nn.Module (flexible)
# ==================================================

class ClassifierNet(nn.Module):
    """A flexible feedforward classifier with skip connections."""

    def __init__(self, input_size, hidden_size, num_classes, dropout=0.3):
        super().__init__()  # ALWAYS call this first!

        # Define layers as attributes
        self.fc1     = nn.Linear(input_size, hidden_size)
        self.fc2     = nn.Linear(hidden_size, hidden_size)
        self.fc3     = nn.Linear(hidden_size, num_classes)
        self.dropout = nn.Dropout(dropout)
        self.bn1     = nn.BatchNorm1d(hidden_size)  # normalize activations
        self.bn2     = nn.BatchNorm1d(hidden_size)

    def forward(self, x):
        """Defines the computation graph. Called when you do model(x)."""
        # Layer 1
        x = self.fc1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout(x)

        # Layer 2 (with residual/skip connection: add input to output)
        identity = x
        x = self.fc2(x)
        x = self.bn2(x)
        x = F.relu(x + identity)  # skip connection: add the input back
        x = self.dropout(x)

        # Output layer (no activation — apply softmax externally or use CrossEntropyLoss)
        x = self.fc3(x)
        return x

model = ClassifierNet(input_size=20, hidden_size=128, num_classes=3)
print(model)  # prints the architecture

x_test = torch.randn(16, 20)  # batch of 16, 20 features
logits = model(x_test)         # forward pass
print(f"\nOutput shape: {logits.shape}")   # [16, 3]
probs  = F.softmax(logits, dim=1)           # convert logits to probabilities
print(f"Probabilities sum to 1: {probs[0].sum().item():.4f}")

## 5. The Training Loop — How PyTorch Models Learn

This is the **heart of PyTorch**. Unlike Keras (which has `model.fit()`), PyTorch gives you explicit control of the training loop. Every iteration follows the same pattern:

```
for each epoch:
    for each mini-batch:
        1. optimizer.zero_grad()   ← clear old gradients
        2. y_pred = model(x_batch) ← forward pass
        3. loss = loss_fn(y_pred, y_batch)  ← compute loss
        4. loss.backward()         ← compute gradients (backprop)
        5. optimizer.step()        ← update weights
```

In [ ]:
# ==================================================
# COMPLETE TRAINING LOOP EXAMPLE
# Binary classification on synthetic spiral data
# ==================================================

# Generate spiral dataset
np.random.seed(42)
N = 500  # per class

def make_spiral(n, c):
    t = np.linspace(0, 4*np.pi, n)
    r = t / (4*np.pi)
    x = r * np.cos(t + c*np.pi) + np.random.randn(n)*0.1
    y = r * np.sin(t + c*np.pi) + np.random.randn(n)*0.1
    return np.stack([x, y], axis=1)

X_np = np.vstack([make_spiral(N, 0), make_spiral(N, 1)]).astype(np.float32)
y_np = np.array([0]*N + [1]*N, dtype=np.int64)

# Convert to PyTorch tensors
X_t = torch.tensor(X_np)
y_t = torch.tensor(y_np)

# Train/test split
idx = torch.randperm(2*N)
split = int(0.8 * 2 * N)
X_train, X_val = X_t[idx[:split]], X_t[idx[split:]]
y_train, y_val = y_t[idx[:split]], y_t[idx[split:]]

# Model
spiral_model = nn.Sequential(
    nn.Linear(2, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 2)   # 2 classes
)

optimizer  = optim.Adam(spiral_model.parameters(), lr=0.01)
loss_fn    = nn.CrossEntropyLoss()  # combines log-softmax + NLL loss
EPOCHS     = 100
BATCH_SIZE = 64

train_dataset = TensorDataset(X_train, y_train)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

train_losses, val_losses = [], []
train_accs,   val_accs   = [], []

for epoch in range(EPOCHS):
    # ---- Training phase ----
    spiral_model.train()   # sets dropout/batchnorm to TRAINING mode
    epoch_loss, correct = 0.0, 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()           # 1. clear gradients
        logits = spiral_model(X_batch)  # 2. forward pass
        loss   = loss_fn(logits, y_batch)  # 3. compute loss
        loss.backward()                 # 4. backpropagation
        optimizer.step()                # 5. update weights

        epoch_loss += loss.item() * len(X_batch)
        correct    += (logits.argmax(1) == y_batch).sum().item()

    train_losses.append(epoch_loss / len(X_train))
    train_accs.append(correct / len(X_train))

    # ---- Validation phase ----
    spiral_model.eval()   # sets dropout/batchnorm to EVAL mode
    with torch.no_grad():  # don't compute gradients during eval (saves memory)
        val_logits = spiral_model(X_val)
        val_loss   = loss_fn(val_logits, y_val).item()
        val_acc    = (val_logits.argmax(1) == y_val).float().mean().item()

    val_losses.append(val_loss)
    val_accs.append(val_acc)

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | "
              f"Train Loss: {train_losses[-1]:.4f}  Acc: {train_accs[-1]:.3f} | "
              f"Val Loss: {val_losses[-1]:.4f}  Acc: {val_accs[-1]:.3f}")

In [ ]:
# Plot training curves and decision boundary
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss curve
axes[0].plot(train_losses, label='Train')
axes[0].plot(val_losses,   label='Validation')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curve'); axes[0].legend()

# Accuracy curve
axes[1].plot(train_accs, label='Train')
axes[1].plot(val_accs,   label='Validation')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Curve'); axes[1].legend()

# Decision boundary
ax = axes[2]
xx, yy = np.meshgrid(np.linspace(-1.5, 1.5, 200), np.linspace(-1.5, 1.5, 200))
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
spiral_model.eval()
with torch.no_grad():
    Z = spiral_model(grid).argmax(1).numpy().reshape(xx.shape)
ax.contourf(xx, yy, Z, alpha=0.4, cmap='RdBu')
ax.scatter(X_np[:N, 0], X_np[:N, 1], c='blue', s=5, alpha=0.5, label='Class 0')
ax.scatter(X_np[N:, 0], X_np[N:, 1], c='red',  s=5, alpha=0.5, label='Class 1')
ax.set_title('Decision Boundary'); ax.legend()

plt.tight_layout()
plt.show()
print(f"Final Val Accuracy: {val_accs[-1]:.3f}")

## 6. Datasets & DataLoaders

For real ML workflows, data doesn't fit in RAM. PyTorch solves this with:
- **`Dataset`**: defines *how* to load one sample (you subclass this)
- **`DataLoader`**: wraps a Dataset and handles batching, shuffling, and parallel loading

In [ ]:
# Custom Dataset — implement __len__ and __getitem__
class TabularDataset(Dataset):
    """Generic tabular dataset that reads from a numpy array."""

    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)   # total number of samples

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]   # return one sample

# Demonstrate with the spiral data
train_ds = TabularDataset(X_np[:800], y_np[:800])
val_ds   = TabularDataset(X_np[800:], y_np[800:])

# DataLoader handles batching + shuffling
train_dl = DataLoader(
    train_ds,
    batch_size=64,
    shuffle=True,       # shuffle every epoch
    num_workers=0,      # parallel data loading workers (set to 2-4 in production)
    pin_memory=False    # set True when using GPU (faster host→GPU transfer)
)
val_dl = DataLoader(val_ds, batch_size=128, shuffle=False)

print(f"Training samples:   {len(train_ds)}")
print(f"Validation samples: {len(val_ds)}")
print(f"Batches per epoch:  {len(train_dl)}")

# Peek at one batch
X_batch, y_batch = next(iter(train_dl))
print(f"\nBatch X shape: {X_batch.shape}")  # [64, 2]
print(f"Batch y shape: {y_batch.shape}")  # [64]

## 7. Saving & Loading Models

In [ ]:
import os, tempfile

save_dir = tempfile.mkdtemp()
model_path = os.path.join(save_dir, 'spiral_model.pth')

# ---- Recommended: save only the state_dict (weights) ----
# The state_dict is a dictionary mapping parameter names to tensors
torch.save(spiral_model.state_dict(), model_path)
print(f"Model saved to: {model_path}")

# Loading: create model with same architecture, then load weights
loaded_model = nn.Sequential(
    nn.Linear(2, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 2)
)
loaded_model.load_state_dict(torch.load(model_path, map_location='cpu'))
loaded_model.eval()
print("Model loaded successfully!")

# Verify predictions match
sample = torch.tensor([[0.5, 0.3]])
with torch.no_grad():
    p1 = spiral_model(sample).argmax().item()
    p2 = loaded_model(sample).argmax().item()
print(f"Original prediction: {p1}, Loaded prediction: {p2}, Match: {p1 == p2}")

# ---- Checkpoint (save full training state for resuming) ----
checkpoint_path = os.path.join(save_dir, 'checkpoint.pth')
torch.save({
    'epoch': EPOCHS,
    'model_state_dict': spiral_model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'val_acc': val_accs[-1],
}, checkpoint_path)

# Loading checkpoint
ckpt = torch.load(checkpoint_path, map_location='cpu')
print(f"\nCheckpoint epoch: {ckpt['epoch']}, Val Acc: {ckpt['val_acc']:.3f}")

## 8. GPU Training

Moving to GPU is simple: move your model and data to the GPU device.

```python
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Move model to GPU
model = model.to(device)

# Move each batch to GPU inside the training loop
for X_batch, y_batch in train_loader:
    X_batch = X_batch.to(device)
    y_batch = y_batch.to(device)
    # ... rest of training loop ...
```

**Rule:** Model and data must be on the **same device**. You cannot mix CPU tensors with GPU tensors.

**Mixed precision (faster GPU training):**
```python
from torch.cuda.amp import autocast, GradScaler
scaler = GradScaler()

with autocast():              # use float16 for forward pass
    output = model(x)
    loss = loss_fn(output, y)

scaler.scale(loss).backward()  # scale to prevent float16 underflow
scaler.step(optimizer)
scaler.update()
```

## 9. Convolutional Neural Networks (CNNs)

CNNs are the go-to architecture for **image data**. They use convolution operations that scan a small filter (kernel) across the image, detecting patterns regardless of where they appear.

**Think of it like:** A CNN looking for a cat's ear doesn't need to know if it's in the top-left or bottom-right — the filter slides everywhere and fires when it finds the pattern.

Key layers:
- `nn.Conv2d(in_channels, out_channels, kernel_size)` — the convolution
- `nn.MaxPool2d(kernel_size)` — downsamples by taking the max in each region
- `nn.BatchNorm2d(num_features)` — normalizes activations (helps training)
- `nn.Flatten()` — converts 2D feature maps to a 1D vector for the classifier head

In [ ]:
# CNN architecture for image classification
# Input: (batch, 1, 28, 28)  ← grayscale 28x28 image (like MNIST)

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # Convolutional feature extractor
        self.features = nn.Sequential(
            # Block 1: (1, 28, 28) → (32, 26, 26) → (32, 13, 13)
            nn.Conv2d(1, 32, kernel_size=3, padding=0),  # 32 filters, 3x3 kernel
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                              # halve spatial dims

            # Block 2: (32, 13, 13) → (64, 11, 11) → (64, 5, 5)
            nn.Conv2d(32, 64, kernel_size=3, padding=0),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 3: (64, 5, 5) → (128, 3, 3)
            nn.Conv2d(64, 128, kernel_size=3, padding=0),
            nn.BatchNorm2d(128),
            nn.ReLU(),
        )

        # Classifier head
        self.classifier = nn.Sequential(
            nn.Flatten(),                 # (128, 3, 3) → (128*3*3=1152,)
            nn.Linear(128 * 3 * 3, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)   # logits for each class
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

cnn = SimpleCNN(num_classes=10)

# Test with a fake batch of MNIST-sized images
fake_batch = torch.randn(8, 1, 28, 28)  # 8 grayscale 28x28 images
output = cnn(fake_batch)
print(f"CNN input shape:  {fake_batch.shape}")   # [8, 1, 28, 28]
print(f"CNN output shape: {output.shape}")         # [8, 10]
print(f"Total CNN parameters: {sum(p.numel() for p in cnn.parameters()):,}")

# Show intermediate feature map shapes
x = fake_batch
for layer in cnn.features:
    x = layer(x)
    print(f"  After {type(layer).__name__:15s}: {x.shape}")

## 10. Recurrent Neural Networks (RNNs / LSTMs)

RNNs process **sequential data** (time series, text, audio) where order matters. The key idea: the network has a **hidden state** that carries information from previous steps.

**Analogy:** Reading a sentence — understanding word 10 depends on remembering words 1-9. An RNN's hidden state is like short-term memory.

**LSTM** (Long Short-Term Memory) improves on basic RNNs by having a "cell state" (long-term memory) and gates that control what to remember and forget.

In [ ]:
# LSTM for time series forecasting

class LSTMForecaster(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,    # features per time step
            hidden_size=hidden_size,  # size of hidden state
            num_layers=num_layers,    # stacked LSTM layers
            batch_first=True,         # input shape: (batch, seq_len, features)
            dropout=0.2 if num_layers > 1 else 0
        )
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        out, (h_n, c_n) = self.lstm(x)  # out: (batch, seq_len, hidden_size)
        # Use the last time step's output for prediction
        last_out = out[:, -1, :]        # (batch, hidden_size)
        pred = self.fc(last_out)        # (batch, output_size)
        return pred

# Example: predict next value in a sequence given past 20 values
lstm = LSTMForecaster(input_size=1, hidden_size=64, num_layers=2, output_size=1)

# Fake batch: 32 sequences, each of length 20, each time step has 1 feature
x_seq = torch.randn(32, 20, 1)
pred  = lstm(x_seq)
print(f"LSTM input shape:  {x_seq.shape}")  # [32, 20, 1]
print(f"LSTM output shape: {pred.shape}")    # [32, 1]  — one value per sequence

# Quick demo: learn a sine wave pattern
np.random.seed(0)
t = np.linspace(0, 100, 2000)
sine = np.sin(t).astype(np.float32)

SEQ_LEN = 30
X_sine = np.array([sine[i:i+SEQ_LEN]   for i in range(len(sine)-SEQ_LEN-1)])
y_sine = np.array([sine[i+SEQ_LEN]     for i in range(len(sine)-SEQ_LEN-1)])
X_sine = X_sine[..., np.newaxis]  # add feature dim: (N, 30, 1)

X_st = torch.tensor(X_sine[:1600]); y_st = torch.tensor(y_sine[:1600])
X_sv = torch.tensor(X_sine[1600:]); y_sv = torch.tensor(y_sine[1600:])

lstm_sine = LSTMForecaster(1, 32, 2, 1)
opt_s = optim.Adam(lstm_sine.parameters(), lr=0.001)
mse_s = nn.MSELoss()

for ep in range(30):
    lstm_sine.train()
    opt_s.zero_grad()
    loss = mse_s(lstm_sine(X_st).squeeze(), y_st)
    loss.backward(); opt_s.step()
    if (ep+1) % 10 == 0:
        lstm_sine.eval()
        with torch.no_grad():
            val_loss = mse_s(lstm_sine(X_sv).squeeze(), y_sv).item()
        print(f"Ep {ep+1}: Train Loss={loss.item():.5f}  Val Loss={val_loss:.5f}")

## 11. Mini Project — Handwritten Digit Classifier (MNIST)

### The Problem

MNIST is the "Hello, World" of deep learning: classify grayscale 28×28 images of handwritten digits (0-9). We'll build and train a CNN that achieves >99% accuracy.

Since downloading MNIST requires internet, we'll simulate the same experiment using synthetic data with similar structure, then show how the real MNIST download would work.

In [ ]:
# ==================================================
# Simulate an MNIST-like classification problem
# using synthetic image patterns
# ==================================================

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split as sk_split

# sklearn's digits dataset — 8x8 handwritten digit images, 10 classes
digits = load_digits()
X_dig = digits.images.astype(np.float32) / 16.0  # normalize to [0, 1]
y_dig = digits.target.astype(np.int64)
print(f"Dataset: {X_dig.shape}  (n_samples, H, W)")
print(f"Classes: {np.unique(y_dig)}")

# Reshape for CNN: (N, C, H, W) — add channel dimension
X_dig = X_dig[:, np.newaxis, :, :]  # (1797, 1, 8, 8)

# Visualize some digits
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_dig[i, 0], cmap='gray_r')
    ax.set_title(str(y_dig[i]))
    ax.axis('off')
plt.suptitle('Sample Digit Images (sklearn digits dataset, 8×8 pixels)', y=1.02)
plt.tight_layout()
plt.show()

# Train/val/test split
X_trv, X_te, y_trv, y_te = sk_split(X_dig, y_dig, test_size=0.15, random_state=42, stratify=y_dig)
X_tr, X_val, y_tr, y_val = sk_split(X_trv, y_trv, test_size=0.15, random_state=42, stratify=y_trv)

# Convert to tensors
X_tr_t  = torch.tensor(X_tr);  y_tr_t  = torch.tensor(y_tr)
X_val_t = torch.tensor(X_val); y_val_t = torch.tensor(y_val)
X_te_t  = torch.tensor(X_te);  y_te_t  = torch.tensor(y_te)

print(f"\nTrain: {X_tr_t.shape}, Val: {X_val_t.shape}, Test: {X_te_t.shape}")

In [ ]:
# CNN for 8x8 digit images
class DigitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Input: (batch, 1, 8, 8)
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)  # (batch, 32, 8, 8)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1) # (batch, 64, 8, 8)
        self.pool  = nn.MaxPool2d(2)                              # (batch, 64, 4, 4)
        self.bn1   = nn.BatchNorm2d(32)
        self.bn2   = nn.BatchNorm2d(64)
        self.dropout = nn.Dropout(0.4)
        self.fc1   = nn.Linear(64 * 4 * 4, 128)
        self.fc2   = nn.Linear(128, 10)    # 10 digit classes

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.dropout(x.view(x.size(0), -1))  # flatten
        x = F.relu(self.fc1(x))
        return self.fc2(x)

digit_model = DigitCNN()
optimizer_d = optim.AdamW(digit_model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler   = optim.lr_scheduler.StepLR(optimizer_d, step_size=20, gamma=0.5)
criterion_d = nn.CrossEntropyLoss()

train_dl_d = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=32, shuffle=True)

EPOCHS_D = 60
train_hist, val_hist = [], []

for epoch in range(EPOCHS_D):
    digit_model.train()
    tr_correct, tr_total = 0, 0
    for Xb, yb in train_dl_d:
        optimizer_d.zero_grad()
        loss = criterion_d(digit_model(Xb), yb)
        loss.backward(); optimizer_d.step()
        tr_correct += (digit_model(Xb).argmax(1) == yb).sum().item()
        tr_total   += len(yb)

    scheduler.step()

    digit_model.eval()
    with torch.no_grad():
        val_acc = (digit_model(X_val_t).argmax(1) == y_val_t).float().mean().item()
    train_hist.append(tr_correct / tr_total)
    val_hist.append(val_acc)

    if (epoch+1) % 15 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS_D} | Train Acc: {train_hist[-1]:.3f} | Val Acc: {val_hist[-1]:.3f}")

# Final test accuracy
digit_model.eval()
with torch.no_grad():
    test_acc = (digit_model(X_te_t).argmax(1) == y_te_t).float().mean().item()
print(f"\n===== Final Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%) =====")

In [ ]:
# Confusion matrix visualization
from sklearn.metrics import confusion_matrix
import seaborn as sns

digit_model.eval()
with torch.no_grad():
    preds = digit_model(X_te_t).argmax(1).numpy()
true = y_te_t.numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
cm = confusion_matrix(true, preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10),
            ax=axes[0])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title(f'Confusion Matrix (Test Acc: {test_acc:.3f})')

# Training curve
axes[1].plot(train_hist, label='Train Accuracy')
axes[1].plot(val_hist,   label='Val Accuracy')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Curve'); axes[1].legend()

plt.tight_layout()
plt.show()

# Show some predictions
print("\nSample Predictions (True → Predicted):")
wrong = [(t, p, i) for i, (t, p) in enumerate(zip(true, preds)) if t != p]
print(f"Misclassified: {len(wrong)}/{len(true)}")
if wrong:
    print("First few errors:", [(t, p) for t, p, _ in wrong[:5]])

## 12. Common Pitfalls

### Pitfall 1: Forgetting `optimizer.zero_grad()`

In PyTorch, gradients **accumulate by default**. If you forget `zero_grad()`, each backward pass adds to the previous gradients, causing incorrect updates.

```python
# WRONG — gradients accumulate across batches
loss.backward()
optimizer.step()

# CORRECT
optimizer.zero_grad()
loss.backward()
optimizer.step()
```

### Pitfall 2: Forgetting `model.eval()` During Inference

Layers like `Dropout` and `BatchNorm` behave **differently during training vs inference**. If you forget `.eval()`, your predictions will be random and inconsistent.

```python
# WRONG — Dropout is active, predictions are stochastic
predictions = model(X_test)

# CORRECT
model.eval()
with torch.no_grad():     # also saves memory by not tracking gradients
    predictions = model(X_test)
```

### Pitfall 3: Shape Mismatch Errors

The most common runtime error. Always print shapes at each step when debugging:
```python
print(x.shape)  # PyTorch prints torch.Size([batch, channels, H, W])
```
Remember: PyTorch convention is `(batch, channels, height, width)` for images.

### Pitfall 4: Wrong Loss Function

| Task | Output | Loss Function |
|---|---|---|
| Binary classification | Sigmoid output | `BCELoss` |
| Binary classification | Raw logits | `BCEWithLogitsLoss` (more stable) |
| Multi-class classification | Raw logits | `CrossEntropyLoss` |
| Regression | Any | `MSELoss` or `L1Loss` |

### Pitfall 5: Not Using `torch.no_grad()` During Evaluation

During evaluation, gradient computation wastes memory and time. Always wrap inference in `torch.no_grad()`.

### Pitfall 6: `.item()` vs Tensor

Always call `.item()` when you want a Python scalar from a single-element tensor (e.g., for logging). Storing tensors in Python lists keeps the entire computation graph alive in memory.

## 13. Interview Q&A

---

**Q1: What is PyTorch's `autograd` and how does it work?**

> Autograd is PyTorch's automatic differentiation engine. When you perform operations on tensors with `requires_grad=True`, PyTorch builds a **dynamic computation graph** (a directed acyclic graph where nodes are tensors and edges are operations). When you call `.backward()`, PyTorch traverses this graph in reverse using the chain rule to compute gradients for every parameter. The gradient is stored in `tensor.grad`. The graph is built fresh at each forward pass ("define-by-run"), which makes it easy to use Python control flow (if/for) inside models.

---

**Q2: PyTorch vs TensorFlow — what are the key differences?**

> **PyTorch:** Define-by-run (eager execution by default), Pythonic debugging with standard print/breakpoints, dominant in research (>70% of papers), more flexible for custom architectures. **TensorFlow:** Originally define-then-run (static graph), now also supports eager execution via `tf.function`. Better ecosystem for production deployment (TensorFlow Serving, TensorFlow Lite, TensorFlow.js). Keras is TensorFlow's high-level API. In 2024, both frameworks support eager execution and deployment, so the choice often comes down to team preference and ecosystem.

---

**Q3: What is the training loop in PyTorch? List the 5 steps.**

> For each mini-batch:
> 1. `optimizer.zero_grad()` — clear accumulated gradients from the previous step
> 2. `y_pred = model(X_batch)` — forward pass to get predictions
> 3. `loss = loss_fn(y_pred, y_batch)` — compute the loss (scalar)
> 4. `loss.backward()` — compute gradients via backpropagation
> 5. `optimizer.step()` — update model weights using the gradients

---

**Q4: What is the difference between `model.train()` and `model.eval()`?**

> Some layers behave differently during training vs inference:
> - **`Dropout`**: during training, randomly zeros elements with probability `p`; during eval, passes all elements unchanged (but scales by `1/(1-p)` to maintain expected values)
> - **`BatchNorm`**: during training, uses the batch's mean/variance for normalization; during eval, uses running statistics accumulated during training
> `model.train()` sets these layers to training mode; `model.eval()` sets them to inference mode. Forgetting `.eval()` is a very common bug.

---

**Q5: What is `nn.Module` and why do you need to call `super().__init__()`?**

> `nn.Module` is the base class for all PyTorch neural network components. It provides:
> - Parameter tracking: any `nn.Parameter` or `nn.Module` attribute is automatically registered
> - `.parameters()` iterator: used by optimizers to update weights
> - `.to(device)`: moves all parameters to a device
> - `.state_dict()` / `load_state_dict()`: for saving/loading
> `super().__init__()` must be called first in `__init__` to initialize this bookkeeping machinery. Without it, PyTorch won't track your parameters.

---

**Q6: What optimizers does PyTorch have and which should you use?**

> Key optimizers in `torch.optim`:
> - **SGD** (with momentum): classic, good for vision with LR scheduling
> - **Adam**: adaptive learning rates, works well out of the box for most tasks
> - **AdamW**: Adam with decoupled weight decay — **default recommendation** for transformers/NLP
> - **RMSprop**: good for RNNs
> 
> For most new projects, start with `AdamW(lr=1e-3, weight_decay=1e-4)` and tune from there.

---

**Q7: What is transfer learning and how would you do it in PyTorch?**

> Transfer learning = using a model pre-trained on a large dataset (e.g., ImageNet) as a starting point for your task. Steps:
> ```python
> import torchvision.models as models
> model = models.resnet50(pretrained=True)   # load pre-trained weights
> 
> # Freeze all layers (don't update during training)
> for param in model.parameters():
>     param.requires_grad = False
> 
> # Replace the final layer for your task (e.g., 5 classes)
> model.fc = nn.Linear(model.fc.in_features, 5)
> # Now only model.fc.parameters() have requires_grad=True
> ```
> This works because early layers learn universal features (edges, textures) that are useful across tasks.

## 14. Resources

### Official
- **Documentation:** https://pytorch.org/docs/stable/
- **Tutorials (official):** https://pytorch.org/tutorials/
- **PyTorch Paper:** https://arxiv.org/abs/1912.01703

### Videos
- **freeCodeCamp — PyTorch for Deep Learning (full course):** https://www.youtube.com/watch?v=V_xro1bcAuA
- **Andrej Karpathy — Neural Networks: Zero to Hero:** https://www.youtube.com/playlist?list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ
- **sentdex — PyTorch tutorials:** https://www.youtube.com/playlist?list=PLQVvvaa0QuDdeMyHEYc0gxFpYwHY2Qfdh

### Books
- **Deep Learning with PyTorch (official book):** https://www.manning.com/books/deep-learning-with-pytorch
- **Dive into Deep Learning (free):** https://d2l.ai/ (PyTorch edition)

### Ecosystem
- **PyTorch Lightning (clean training loops):** https://lightning.ai/docs/pytorch/stable/
- **HuggingFace (transformers/pre-trained models):** https://huggingface.co/
- **torchvision (image datasets + models):** https://pytorch.org/vision/stable/
- **torchaudio (audio processing):** https://pytorch.org/audio/stable/

## 15. Summary & What's Next

### What You Learned

| Concept | Key Takeaway |
|---|---|
| **Tensors** | n-dimensional arrays that live on CPU/GPU and track gradients |
| **Autograd** | Automatically computes gradients via dynamic computation graph + `.backward()` |
| **nn.Module** | Base class for all models; define layers in `__init__`, computation in `forward()` |
| **Training loop** | zero_grad → forward → loss → backward → step (5 steps, every batch) |
| **Dataset/DataLoader** | Custom data loading via `__len__` / `__getitem__`; automatic batching/shuffling |
| **model.train() / eval()** | Switch Dropout/BatchNorm behavior; always use `.eval()` + `no_grad()` for inference |
| **CNN** | Convolution+pooling for images; `(batch, channels, H, W)` tensor convention |
| **LSTM** | Recurrent network for sequences; `(batch, seq_len, features)` with `batch_first=True` |
| **Saving** | Save `state_dict()`, not the full model, for portability |

### What's Next

Now you understand PyTorch's building blocks. The next notebooks cover:
- **TensorFlow** — Google's production deep learning platform with `tf.function` and TensorFlow Serving
- **Keras** — the high-level API on top of TensorFlow with `model.fit()`, simplifying training
- **JAX** — NumPy + automatic differentiation + JIT compilation, used by Google DeepMind